# 01 Data Exploration

Notebook này khám phá corpus lao động đã commit và thử rule lọc trước khi đưa logic ổn định vào `src/data/`.

Mặc định notebook chạy offline trên `data/processed/labor_corpus.jsonl`. Download public dataset là bước tùy chọn vì file parquet nguồn khá lớn. Notebook không ghi đè corpus production.

In [ ]:
import html
import json
import re
from pathlib import Path

import pandas as pd
from datasets import get_dataset_config_names, load_dataset

DATASET_NAME = "th1nhng0/vietnamese-legal-documents"
DOWNLOAD_DATASET = False
DATA_SPLIT = "data[:2000]"  # Change to "data" for full exploration.
EXPORT_CORPUS = False
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LOCAL_CORPUS = PROJECT_ROOT / "data/processed/labor_corpus.jsonl"
NOTEBOOK_OUTPUT = PROJECT_ROOT / "data/processed/labor_corpus_notebook.jsonl"

print("mode:", "download public dataset" if DOWNLOAD_DATASET else "offline committed corpus")

In [ ]:
if DOWNLOAD_DATASET:
    print("configs:", get_dataset_config_names(DATASET_NAME))
    print("split:", DATA_SPLIT)
    metadata_rows = [dict(row) for row in load_dataset(DATASET_NAME, "metadata", split=DATA_SPLIT)]
    content_rows = [dict(row) for row in load_dataset(DATASET_NAME, "content", split=DATA_SPLIT)]
    content_map = {str(row["id"]): row.get("content_html", "") for row in content_rows}
    documents = [{**row, "id": str(row["id"]), "content_html": content_map.get(str(row["id"]), "")} for row in metadata_rows if str(row["id"]) in content_map]
else:
    documents = [json.loads(line) for line in LOCAL_CORPUS.read_text(encoding="utf-8").splitlines() if line.strip()]

print(f"documents={len(documents)}")

In [ ]:
TITLE_KEYWORDS = ["lao động", "việc làm", "bảo hiểm xã hội", "bảo hiểm thất nghiệp", "tiền lương", "lương tối thiểu", "an toàn vệ sinh lao động", "dạy nghề", "đào tạo nghề", "công đoàn"]
STRONG_PHRASES = ["hợp đồng lao động", "bộ luật lao động", "người lao động", "người sử dụng lao động", "bảo hiểm thất nghiệp", "trợ cấp thôi việc", "trợ cấp mất việc", "kỷ luật lao động", "tranh chấp lao động", "thỏa ước lao động tập thể", "an toàn vệ sinh lao động", "giấy phép lao động", "làm thêm giờ"]
SUPPORTING_TERMS = ["lương tối thiểu", "tiền lương", "học nghề", "đào tạo nghề", "bảo hiểm xã hội", "tai nạn lao động", "bệnh nghề nghiệp", "công đoàn", "đình công", "lao động nữ", "thai sản", "lao động nước ngoài"]


def normalize_text(text: str | None) -> str:
    text = html.unescape(text or "")
    text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).lower().strip()


def is_labor_related(document: dict) -> bool:
    title_fields = " ".join(str(document.get(field) or "") for field in ["title", "nganh", "linh_vuc", "thong_tin_ap_dung"])
    title_text = normalize_text(title_fields)
    full_text = normalize_text(f"{title_fields} {document.get('content_html', '')}")
    if any(keyword in title_text for keyword in TITLE_KEYWORDS):
        return True
    if any(phrase in full_text for phrase in STRONG_PHRASES):
        return True
    hits = sum(term in full_text for term in SUPPORTING_TERMS)
    return hits >= 3 and "lao động" in full_text


labor_documents = [document for document in documents if is_labor_related(document)]
print(f"labor documents={len(labor_documents)}")

In [ ]:
preview = pd.DataFrame([
    {
        "id": document["id"],
        "title": document.get("title"),
        "type": document.get("loai_van_ban"),
        "effective_date": document.get("ngay_co_hieu_luc"),
        "text_preview": normalize_text(document.get("content_html"))[:180],
    }
    for document in labor_documents[:10]
])
preview

In [ ]:
if EXPORT_CORPUS:
    NOTEBOOK_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with NOTEBOOK_OUTPUT.open("w", encoding="utf-8") as output:
        for document in labor_documents:
            output.write(json.dumps(document, ensure_ascii=False) + "\n")
    print(f"wrote {len(labor_documents)} documents to {NOTEBOOK_OUTPUT}")
else:
    print("EXPORT_CORPUS=False: exploration completed without writing files.")

## Production corpus

Sau khi kiểm tra rule lọc trong notebook, dùng CLI production để tải nguồn public và tạo corpus đầy đủ:

```bash
python -m src.data.build_corpus
```